# High-Throughput Processing using Mofography and Dask
In this notebook, 

- we'll load up a dataset of MOFs, such as the IZA database, 
- create ASE atoms objects from it,
- create a workflow for a single materials, and, 
- extend and parallelize it over the whole dataset using Dask. 

This notebook uses a 'local' cluster, meaning your personal laptop or desktop with a single CPU, however, the same code can be used to scale up to a cluster with many CPUs or GPUs. Please refer the Dask documentation for more information on how to set up a cluster here: https://docs.dask.org

## Imports and Paths

Let's import the necessary libraries and set some paths

In [ ]:
#* General imports
import os, sys
import numpy as np 
from pathlib import Path 
import pandas as pd
from tqdm import tqdm

#* Add the path to mofography to the system path
path_to_mgr = Path('C:/Users/Arun Gopalan/OneDrive - The University of Manchester/my_cloned_repos/mofography')
sys.path.append(str(path_to_mgr))


import mofography as mgr

#* Needed for cheminformatics if any
from ase.io import read, write
from ase.build import make_supercell 
from ase.geometry import complete_cell


#* for plotting
import plotly.graph_objects as go

#* Needed for the workflow
import dask
from dask import delayed
from dask.distributed import Client, progress, LocalCluster

## Load the dataset 

I have the dataset saved up, if you don't have it, check out hte getting data from MOFDB tutuorial to create and save the dataframe.

In [ ]:
iza_df = pd.read_feather('C:/Users/Arun Gopalan/OneDrive - The University of Manchester/my_cloned_repos/mofography/examples/iza_df.feather')

the dataset looks like this:

In [ ]:
print(iza_df.columns)
iza_df.head(10)

In [ ]:
## Create ASE atoms objects from the dataframe

In [ ]:
iza_atoms = mgr.atoms_from_mofdb_df(iza_df)

## Create a workflow for a single material
Let's pick one random material and define our usual workflow where we create pore graphs from the atoms object. If this looks unfamiliar, please refer to the previous tutorials.

In [ ]:
mat_atoms = iza_atoms[3] # you can try other materials also

In [ ]:
# compute bonds
mat_updated  = mgr.update_with_covalent_bonds(mat_atoms)
dgrid = mgr.dgrid_from_atoms_cpu(mat_atoms, spacing=0.5) #* compute distance grid
shape = dgrid.shape #* we'll need this for later
regions, maxima = mgr.regions_from_dgrid(dgrid, mask_thickness=1.0) #* compute regions
maxima_radii = dgrid[tuple(maxima.T)] #* grab the radii of the maxima from dgrid
regions_pbc  = mgr.apply_pbc(regions, maxima, maxima_radii, mat_atoms, return_conn=False) #* apply PBC and return as 3D array
pbc_groups = mgr.apply_pbc(regions, maxima, maxima_radii, mat_atoms, return_conn=True) #* return as connected groups
rag = mgr.get_rag_from_regions_and_grid(regions, dgrid, mat_atoms, maxima) #* create pore graph
rag = mgr.add_maxima_to_rag(rag, maxima, maxima_radii, shape, mat_atoms) #* add maxima to the graph
rag = mgr.add_pbc_to_rag(rag, pbc_groups) #* add PBC information to the graph
connections = mgr.get_connections_for_rag(rag, regions, maxima, mat_atoms, dgrid) #* compute windows

In [ ]:
rag.graph['mat_atoms'] = mat_atoms
rag.graph['dgrid'] = dgrid
rag.graph['regions'] = regions
rag.graph['maxima'] = maxima
rag.graph['maxima_radii'] = maxima_radii
rag.graph['regions_pbc'] = regions_pbc
rag.graph['pbc_groups'] = pbc_groups
rag.graph['connections'] = connections

## Visualize 
Let's visualize just this material once very quickly to make sure we are on the right track, before we parallelize the workflow.

In [ ]:
frame_trace = mgr.plot_framework_as_mesh(mat_updated, stylize=True)
cell_trace = mgr.plot_cell(mat_updated)
maxima_trace = mgr.plot_maxima(dgrid, maxima, mat_atoms)
surface_regions = mgr.plot_surface_dgrid(dgrid, isovalue=1.5, ase_atoms = mat_atoms, region_labels=regions, stylize=True)
rag_trace = mgr.plot_rag(rag)

fig = go.Figure(data=[*frame_trace, cell_trace, maxima_trace, surface_regions, rag_trace], layout=mgr.epic_layout)
fig.update_scenes(aspectmode='data')
fig.show()

# Define workflow using Dask delayed
Dask delayed essentially defines the compute graph for the workflow, without actually runnning it right away. This is very good for two reasons:
- We can parallelize the workflow very well. Not just over materials, but, over the steps needed for each material also can be parallelized. 
- If one the calculations fail, it won't affect the rest of the materials
This in only a quick and dirty use of Dask without any of the nuances explained in detail. 

We can have another detailed intro tutorial on dask in the future, but, the current dask documentation is very good and I highly recommend it.
 However, there are a couple ofthings we need to keep in mind, dask delayed objects are immutable, hence, we can't work with them like tuples like how we did above. But, this is an easy fix, we just have to wrap the couple of steps into a function like this : 

In [ ]:
def regions_maxima_radii_from_dgrid(dgrid, mask_thickness = 1.0):
    regions, maxima = mgr.regions_from_dgrid(dgrid, mask_thickness = mask_thickness)
    maxima_radii = dgrid[tuple(maxima.T )]
    return regions, maxima, maxima_radii

def make_rag(regions, dgrid, maxima, maxima_radii, mat_atoms):
    rag = mgr.get_rag_from_regions_and_grid(regions, dgrid, mat_atoms)
    shape = dgrid.shape
    rag = mgr.add_maxima_to_rag(rag, maxima, maxima_radii, shape, mat_atoms)
    rag.graph['dgrid'] = dgrid
    rag.graph['regions'] = regions
    rag.graph['maxima'] = maxima
    rag.graph['maxima_radii'] = maxima_radii
    rag.graph['mat_atoms'] = mat_atoms
    return rag

def connections_for_rag(rag):
    connections = mgr.get_connections_for_rag(rag, rag.graph['regions'], rag.graph['maxima'], rag.graph['mat_atoms'], rag.graph['dgrid'])
    rag.graph['connections'] = connections
    return rag

Now, we can use these functions to define the workflow using dask delayed.

In [ ]:
delayed_rag_futures = [] 

for mat_atoms in iza_atoms:
    dgrid = delayed(mgr.dgrid_from_atoms_cpu)(mat_atoms, spacing=0.5, precompute=True)
    regions_maxima_radii = delayed(regions_maxima_radii_from_dgrid)(dgrid, mask_thickness=0.5)
    regions_pbc = delayed(mgr.apply_pbc)(regions_maxima_radii[0], regions_maxima_radii[1], regions_maxima_radii[2], mat_atoms)
    rag = delayed(make_rag)(regions_maxima_radii[0], dgrid, regions_maxima_radii[1],regions_maxima_radii[2], mat_atoms)
    rag = delayed(connections_for_rag)(rag)
    delayed_rag_futures.append(rag)

In [ ]:
#* try running one to see if it works
delayed_rag_futures[0].compute()

## Start a dask cluster locally. 
Look at the dask documentation for more information on how to set up a cluster on a remote machine or a cloud service or to change the number of workers, processes, threads etc. 

In [ ]:
cluster = LocalCluster()
client = Client(cluster)

print('THe client is running on : ', client.dashboard_link)

IF you go to the link, you'll see something like this:  https://docs.dask.org/en/stable/dashboard.html. You can see the progress of the workflow and the resources being used.

## Persist 
Submit the jobs to run in the background, this works on a local clusterin and on a HPC.

In [ ]:
#* submit the workflow using the futures API

futures = dask.persist(*delayed_rag_futures) 

In [ ]:
#* quick progress tracking, run this cell again for updates after any errors

progress(futures)

## Grab the results

Please wait for the above calculations to finish. After the workflow is done, we can grab the results like this

In [ ]:
results = client.compute(futures)

grab the results for finished futures. If some are still running or errored, let's set them to None

In [ ]:
rags = [r.result() if r.status == 'finished' else None for r in results]

Check how many of them failed


In [ ]:

(np.array(rags) == None).sum()



Notice that some of them failed, this is OK, it is just to show how to handle errors (the updated routines don't fail as much :)). Let's just redefine the workflow with updated functions where will compute the edges and windows first and then make a graph from that, instead of using RAG functions from skimage. 

## Redefine workflow
Redifine the workflow with the newer functions in mgr, that are less prone to error. Details in later cells.

Now, let's redefine the workflow using dask delayed like before, notice the change in the order of the calculations 
before: dgrid -> regions -> maxima -> regions_pbc -> rag -> connections -> rest of the steps
after: dgrid -> regions -> maxima -> connections -> rag-> rest of the steps

In [ ]:
def make_rag_updated(regions, dgrid, maxima,maxima_radii, mat_atoms):
    connections = mgr.connections_from_regions_and_dgrid(regions, dgrid, maxima, mat_atoms)
    rag = mgr.rag_from_connections(connections, maxima)
    shape = dgrid.shape
    rag = mgr.add_maxima_to_rag(rag, maxima, maxima_radii, shape, mat_atoms)
    rag.graph['dgrid'] = dgrid
    rag.graph['regions'] = regions
    rag.graph['maxima'] = maxima
    rag.graph['ase_atoms'] = mat_atoms
    rag.graph['connections'] = connections

    return rag

delayed_rag_futures = [] 

for mat_atoms in iza_atoms:
    dgrid = delayed(mgr.dgrid_from_atoms_cpu)(mat_atoms, spacing=0.5, precompute=True)
    # dgrid = delayed(mgr.zoom_grid)(dgrid, 2.0) #* let's zoom the grid make it finer using interpolation
    regions_maxima_radii = delayed(regions_maxima_radii_from_dgrid)(dgrid, mask_thickness=0.5)
    regions_pbc = delayed(mgr.apply_pbc)(regions_maxima_radii[0], regions_maxima_radii[1], regions_maxima_radii[2], mat_atoms)
    #* notice the change in the wrapping function here, defined above
    rag = delayed(make_rag_updated)(regions_maxima_radii[0], dgrid, regions_maxima_radii[1], regions_maxima_radii[2], mat_atoms)
    delayed_rag_futures.append(rag)

In [ ]:
delayed_rag_futures[0].compute()

## Re-start cluster
You can shutdown and restart the cluster also `client.shutdown()` and `client.restart()` if you want to clear the memory and start fresh.

In [ ]:
client = Client(LocalCluster())

print('THe client is running on : ', client.dashboard_link)

## Persist 
Submit the pore graphs calculations to run in the background

In [ ]:
futures= dask.persist(*delayed_rag_futures)

In [ ]:
progress(futures)

## Collect the results
Please make sure you wait for a the calculations to finish before you collect the results.

In [ ]:
results = client.compute(futures)
rags = [r.result() if r.status == 'finished' else None for r in results]

Let's see how many failed th

In [ ]:
np.sum([r is not None for r in rags])

In [ ]:
rags

## Conclusions and Exercise 
We saw a guick HTS calculation where we grabbed the IZA dataset from MOFDB and calculated the pore graphs for all the materials in the dataset. We used Dask to parallelize the workflow and ran it on a local cluster. We also saw how to handle errors and re-run the workflow with less error prone functions.

Exercise:
As an exercise, try to run this on an HPC on a larger dataset, and/or try to add additional properties to the pore graphs based on the other tutorials.